# COVID City Daily - Data Analysis

This notebook cleans the COVID city daily dataset, answers the main questions, and adds summary charts for cases, deaths, and vaccines.

In [ ]:
import pandas as pd

try:
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("Plotly is not installed. Charts will be skipped. Run: pip install plotly")

In [ ]:
LOCAL_PATH = "data/covid_city_daily.csv"
REMOTE_URL = "https://raw.githubusercontent.com/danabadarneh/Data-Analysis-GSG/main/DATA%20GSG/COVID%20-%20City%20-%20Daily.csv"

try:
    raw_df = pd.read_csv(LOCAL_PATH, na_values=["."])
    print(f"Loaded local file: {LOCAL_PATH}")
except FileNotFoundError:
    raw_df = pd.read_csv(REMOTE_URL, na_values=["."])
    print("Loaded data from GitHub")

raw_df.head()

## 1. Data Cleaning

- Treat `.` as missing data.
- Build a real `date` column from `year`, `month`, and `day`.
- Convert numeric columns from text to numbers.
- Fill missing numeric values with `0` so calculations work correctly.

In [ ]:
df = raw_df.copy()

df = df.drop_duplicates()
df["date"] = pd.to_datetime(df[["year", "month", "day"]])

numeric_cols = [col for col in df.columns if col not in ["year", "month", "day", "cityid", "date"]]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

summary = pd.DataFrame({
    "rows": [df.shape[0]],
    "columns": [df.shape[1]],
    "cities": [df["cityid"].nunique()],
    "start_date": [df["date"].min().date()],
    "end_date": [df["date"].max().date()],
})

summary

## 2. Main KPIs

In [ ]:
kpis = pd.DataFrame({
    "metric": ["Total new cases", "Total new deaths", "Total new vaccines"],
    "value": [
        df["new_case_count"].sum(),
        df["new_death_count"].sum(),
        df["new_vaccine_count"].sum(),
    ],
})

kpis["value"] = kpis["value"].astype("int64")
kpis

## Q1. Calculating the Number of Deaths in a Period

Example period: from `2020-01-01` to before `2020-04-01`.

In [ ]:
start_date = pd.Timestamp("2020-01-01")
end_date = pd.Timestamp("2020-04-01")

period_df = df[(df["date"] >= start_date) & (df["date"] < end_date)]
total_deaths_in_period = int(period_df["death_count"].sum())

print(f"Total deaths from {start_date.date()} to before {end_date.date()}: {total_deaths_in_period:,}")

## Q2. How Many New Vaccines Are There on a Specific Date?

In [ ]:
selected_date = pd.Timestamp("2021-04-16")

vaccines_on_date = int(df.loc[df["date"] == selected_date, "new_vaccine_count"].sum())
print(f"Total new vaccines on {selected_date.date()}: {vaccines_on_date:,}")

## Q3. How Many Cases Appeared in a City by ID?

Use `new_case_count` for appeared/new cases. Using `case_count` would double-count cumulative totals across many days.

In [ ]:
city_id = 5

city_cases = int(df.loc[df["cityid"] == city_id, "new_case_count"].sum())
print(f"Total new cases in city ID {city_id}: {city_cases:,}")

## 3. Top Cities by New Cases

In [ ]:
top_cities = (
    df.groupby("cityid", as_index=False)["new_case_count"]
    .sum()
    .sort_values("new_case_count", ascending=False)
    .head(10)
)

top_cities["new_case_count"] = top_cities["new_case_count"].astype("int64")
top_cities

In [ ]:
if HAS_PLOTLY:
    fig = px.bar(
        top_cities,
        x="cityid",
        y="new_case_count",
        title="Top 10 Cities by New Cases",
        labels={"cityid": "City ID", "new_case_count": "New cases"},
    )
    fig.show()

## 4. Daily Trends

In [ ]:
daily = (
    df.groupby("date", as_index=False)
    .agg(
        new_cases=("new_case_count", "sum"),
        new_deaths=("new_death_count", "sum"),
        new_vaccines=("new_vaccine_count", "sum"),
    )
)

peak_cases = daily.loc[daily["new_cases"].idxmax()]
peak_deaths = daily.loc[daily["new_deaths"].idxmax()]
peak_vaccines = daily.loc[daily["new_vaccines"].idxmax()]

pd.DataFrame({
    "metric": ["Peak new cases", "Peak new deaths", "Peak new vaccines"],
    "date": [peak_cases["date"].date(), peak_deaths["date"].date(), peak_vaccines["date"].date()],
    "value": [int(peak_cases["new_cases"]), int(peak_deaths["new_deaths"]), int(peak_vaccines["new_vaccines"])],
})

In [ ]:
if HAS_PLOTLY:
    fig = px.line(
        daily,
        x="date",
        y="new_cases",
        title="Daily New Cases Over Time",
        labels={"date": "Date", "new_cases": "New cases"},
    )
    fig.show()

    fig = px.line(
        daily,
        x="date",
        y="new_deaths",
        title="Daily New Deaths Over Time",
        labels={"date": "Date", "new_deaths": "New deaths"},
    )
    fig.show()

    fig = px.line(
        daily,
        x="date",
        y="new_vaccines",
        title="Daily New Vaccines Over Time",
        labels={"date": "Date", "new_vaccines": "New vaccines"},
    )
    fig.show()

## 5. Conclusion

- The dataset contains daily COVID records for 53 cities from 2020-01-01 to 2023-06-05.
- The highest daily new cases happened on 2022-01-16.
- The highest daily new deaths happened on 2020-04-12.
- The highest daily new vaccines happened on 2021-04-16.
- City ID 1 has the highest total number of new cases in the dataset.